# Stacked Ensemble on 352 L1-Selected Categorical Features

This notebook trains stacked ensemble models using the 352 features selected by the L1 feature-selection run.

- External split: 75% train / 25% test only
- Meta-feature generation: 5-fold stratified OOF on the training set only
- Base-model combinations are hardcoded in this notebook
- Meta-model: Logistic Regression
- Meta input: soft probability outputs from the base learners
- Data source: `combined_embeddings/categorical_embeddings_train.csv` and `combined_embeddings/categorical_embeddings_test.csv`

In [1]:
import json
import os
import random
from copy import deepcopy
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

ROOT = Path('/Users/parvgoyal/IP/Multitaste-Model')
EMB_DIR = ROOT / 'combined_embeddings'
FS_DIR = ROOT / 'fs_results'
L1_RESULTS_PATH = FS_DIR / 'categorical_feature_selection_L1_352.csv'
TRAIN_PATH = EMB_DIR / 'categorical_all_feature_2197_train.csv'
TEST_PATH = EMB_DIR / 'categorical_all_feature_2197_test.csv'
RESULTS_PATH = FS_DIR / 'stacked_l1_results.csv'
BEST_PRED_PATH = FS_DIR / 'stacked_l1_best_predictions.csv'
BEST_CONFIG_PATH = FS_DIR / 'stacked_l1_best_config.json'
BEST_MODEL_PATH = FS_DIR / 'stacked_l1_best_bundle.joblib'

LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
CLASS_NAMES = LABEL_COLS
NUM_CLASSES = len(CLASS_NAMES)
print('Paths ready:', TRAIN_PATH, TEST_PATH, L1_RESULTS_PATH)

Paths ready: /Users/parvgoyal/IP/Multitaste-Model/combined_embeddings/categorical_all_feature_2197_train.csv /Users/parvgoyal/IP/Multitaste-Model/combined_embeddings/categorical_all_feature_2197_test.csv /Users/parvgoyal/IP/Multitaste-Model/fs_results/categorical_feature_selection_L1_352.csv


In [2]:
# Load the L1 feature list and the categorical train/test splits
l1_df = pd.read_csv(L1_RESULTS_PATH)
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

stack_combinations = [
    'BiLSTM + XGBoost + SVM',
    'BiLSTM + Random Forest + XGBoost',
    'BiLSTM + Extra Trees + XGBoost',
    'Random Forest + XGBoost + SVM',
    'Extra Trees + XGBoost + SVM',
    'Random Forest + Extra Trees + XGBoost',
    'Random Forest + Extra Trees + SVM',
    'BiLSTM + Random Forest + Extra Trees',
    'BiLSTM + Random Forest + SVM',
    'BiLSTM + Extra Trees + SVM',
    'BiLSTM + Random Forest + XGBoost + SVM',
    'BiLSTM + Extra Trees + XGBoost + SVM',
    'BiLSTM + Random Forest + Extra Trees + XGBoost',
    'Random Forest + Extra Trees + XGBoost + SVM',
    'BiLSTM + Random Forest + Extra Trees + SVM',
]

selected_feature_names = [name.strip() for name in str(l1_df.loc[0, 'Selected_Feature_Names']).split('|') if name.strip()]
missing_features = [name for name in selected_feature_names if name not in train_df.columns]

print('L1 rows:', len(l1_df))
print('Selected features parsed:', len(selected_feature_names))
print('Missing features:', len(missing_features))
print('Base-model combinations (hardcoded):', len(stack_combinations))

if missing_features:
    raise ValueError(f'Missing selected features in categorical train CSV: {missing_features[:10]}')

X_train = train_df[selected_feature_names].to_numpy(dtype=np.float32)
y_train = train_df[LABEL_COLS].to_numpy().argmax(axis=1)
X_test = test_df[selected_feature_names].to_numpy(dtype=np.float32)
y_test = test_df[LABEL_COLS].to_numpy().argmax(axis=1)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train distribution:', np.bincount(y_train, minlength=NUM_CLASSES))
print('y_test distribution:', np.bincount(y_test, minlength=NUM_CLASSES))

L1 rows: 10
Selected features parsed: 352
Missing features: 0
Base-model combinations (hardcoded): 15
X_train shape: (13365, 352)
X_test shape: (4456, 352)
y_train distribution: [8105 2133  432 1155 1540]
y_test distribution: [2703  711  144  385  513]


In [ ]:
def macro_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 0:
        return 0.0
    specificities = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specificities))


def compute_metrics(y_true, y_pred, y_proba):
    metrics = {
        'ACC': accuracy_score(y_true, y_pred),
        'BACC': balanced_accuracy_score(y_true, y_pred),
        'PRE': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'SPEC': macro_specificity(y_true, y_pred),
        'SENS': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
    }
    try:
        metrics['AUC'] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        metrics['AUC'] = float('nan')
    return metrics


class BiLSTMClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_dim=64, num_layers=1, dropout=0.2, epochs=8, batch_size=64, lr=1e-3, seed=42):
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.seed = seed

    def _set_seed(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)

    def _build_model(self, input_dim, num_classes):
        class _Net(nn.Module):
            def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_classes):
                super().__init__()
                self.rnn = nn.LSTM(
                    input_size=input_dim,
                    hidden_size=hidden_dim,
                    num_layers=num_layers,
                    batch_first=True,
                    bidirectional=True,
                    dropout=dropout if num_layers > 1 else 0.0,
                )
                self.dropout = nn.Dropout(dropout)
                self.fc = nn.Linear(hidden_dim * 2, num_classes)

            def forward(self, x):
                out, _ = self.rnn(x)
                out = out[:, -1, :]
                out = self.dropout(out)
                return self.fc(out)

        return _Net(input_dim, self.hidden_dim, self.num_layers, self.dropout, num_classes)

    def fit(self, X, y):
        self._set_seed()
        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.num_classes_ = len(self.classes_)

        class_counts = np.bincount(y, minlength=self.num_classes_).astype(float)
        class_counts[class_counts == 0.0] = 1.0
        class_weights = len(y) / (self.num_classes_ * class_counts)

        X_train_t = torch.tensor(X_scaled, dtype=torch.float32).unsqueeze(-1)
        y_train_t = torch.tensor(y, dtype=torch.long)
        train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
        train_loader = torch.utils.data.DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_ = self._build_model(input_dim=1, num_classes=self.num_classes_).to(device)
        class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_t)
        optimizer = optim.Adam(self.model_.parameters(), lr=self.lr)

        for _ in range(self.epochs):
            self.model_.train()
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                optimizer.zero_grad()
                loss = criterion(self.model_(xb), yb)
                loss.backward()
                optimizer.step()

        self.device_ = device
        return self

    def predict_proba(self, X):
        X_scaled = self.scaler_.transform(X)
        X_t = torch.tensor(X_scaled, dtype=torch.float32).unsqueeze(-1)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_t.to(self.device_))
            proba = torch.softmax(logits, dim=1).cpu().numpy()
        return proba

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def make_base_estimator(name, seed=42):
    if name == 'Random Forest':
        return RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=1, class_weight='balanced')
    if name == 'Extra Trees':
        return ExtraTreesClassifier(n_estimators=300, random_state=seed, n_jobs=1, class_weight='balanced')
    if name == 'XGBoost':
        return 
        
    if name == 'SVM':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='rbf', probability=True, decision_function_shape='ovr', class_weight='balanced', random_state=seed)),
        ])
    if name == 'BiLSTM':
        return BiLSTMClassifier(seed=seed)
    raise ValueError(f'Unknown base model: {name}')


def fit_with_class_weight(estimator, X, y, model_name):
    sample_weight = compute_sample_weight(class_weight='balanced', y=y)

    if model_name == 'XGBoost':
        estimator.fit(X, y, sample_weight=sample_weight)
        return estimator

    if model_name == 'SVM':
        estimator.fit(X, y, svm__sample_weight=sample_weight)
        return estimator

    try:
        estimator.fit(X, y, sample_weight=sample_weight)
    except TypeError:
        estimator.fit(X, y)
    return estimator


def parse_combo(combo_name):
    return [part.strip() for part in combo_name.split('+') if part.strip()]


print('Base estimators available: Random Forest, Extra Trees, XGBoost, SVM, BiLSTM')

Base estimators available: Random Forest, Extra Trees, XGBoost, SVM, BiLSTM


In [4]:
def generate_oof_and_test_proba(model_name, X_tr, y_tr, X_te, cv, seed):
    oof_proba = np.zeros((len(y_tr), NUM_CLASSES), dtype=np.float32)
    fold_test_probs = []

    for fold_idx, (fit_idx, val_idx) in enumerate(cv.split(X_tr, y_tr), start=1):
        X_fit, y_fit = X_tr[fit_idx], y_tr[fit_idx]
        X_val = X_tr[val_idx]

        fold_model = make_base_estimator(model_name, seed=seed + fold_idx)
        fold_model = fit_with_class_weight(fold_model, X_fit, y_fit, model_name)

        oof_proba[val_idx] = fold_model.predict_proba(X_val)
        fold_test_probs.append(fold_model.predict_proba(X_te))

    mean_test_proba = np.mean(np.stack(fold_test_probs, axis=0), axis=0)

    final_model = make_base_estimator(model_name, seed=seed)
    final_model = fit_with_class_weight(final_model, X_tr, y_tr, model_name)

    return oof_proba, mean_test_proba, final_model


def fit_and_score_combination(combo_name, seed=42, cv_splits=5):
    base_names = parse_combo(combo_name)
    missing = [name for name in base_names if name not in {'Random Forest', 'Extra Trees', 'XGBoost', 'SVM', 'BiLSTM'}]
    if missing:
        raise ValueError(f'Unsupported base model names in {combo_name}: {missing}')

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    train_meta_parts = []
    test_meta_parts = []
    fitted_base_models = []

    for base_name in base_names:
        oof_proba, test_proba, final_model = generate_oof_and_test_proba(
            model_name=base_name,
            X_tr=X_train,
            y_tr=y_train,
            X_te=X_test,
            cv=cv,
            seed=seed,
        )
        train_meta_parts.append(oof_proba)
        test_meta_parts.append(test_proba)
        fitted_base_models.append((base_name, final_model))

    train_meta = np.hstack(train_meta_parts)
    test_meta = np.hstack(test_meta_parts)

    meta_model = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=4000, random_state=seed, class_weight='balanced')),
    ])
    meta_model.fit(train_meta, y_train)

    train_pred = meta_model.predict(train_meta)
    test_pred = meta_model.predict(test_meta)
    train_proba = meta_model.predict_proba(train_meta)
    test_proba = meta_model.predict_proba(test_meta)

    metrics = {
        'Train': compute_metrics(y_train, train_pred, train_proba),
        'Test': compute_metrics(y_test, test_pred, test_proba),
    }

    return {
        'Stack-base Model': combo_name,
        'Base_Models': ' + '.join(base_names),
        'Selected_Features': len(selected_feature_names),
        'train_meta': train_meta,
        'test_meta': test_meta,
        'train_pred': train_pred,
        'test_pred': test_pred,
        'train_proba': train_proba,
        'test_proba': test_proba,
        'meta_model': meta_model,
        'fitted_base_models': fitted_base_models,
        'metrics': metrics,
    }


results = []
best_bundle = None
best_score = -np.inf

for combo_name in stack_combinations:
    print(f'Running stack: {combo_name}')
    bundle = fit_and_score_combination(combo_name, seed=SEED, cv_splits=5)
    row = {
        'Stack-base Model': bundle['Stack-base Model'],
        'Base_Models': bundle['Base_Models'],
        'Selected_Features': bundle['Selected_Features'],
    }
    for prefix, metric_dict in bundle['metrics'].items():
        for metric_name, metric_value in metric_dict.items():
            row[f'{prefix}_{metric_name}'] = round(float(metric_value), 4)
    results.append(row)

    score = bundle['metrics']['Test']['BACC']
    if score > best_score:
        best_score = score
        best_bundle = bundle

results_df = pd.DataFrame(results).sort_values(by='Test_BACC', ascending=False).reset_index(drop=True)
results_df

Running stack: BiLSTM + XGBoost + SVM
Running stack: BiLSTM + Random Forest + XGBoost
Running stack: BiLSTM + Extra Trees + XGBoost
Running stack: Random Forest + XGBoost + SVM
Running stack: Extra Trees + XGBoost + SVM
Running stack: Random Forest + Extra Trees + XGBoost
Running stack: Random Forest + Extra Trees + SVM
Running stack: BiLSTM + Random Forest + Extra Trees
Running stack: BiLSTM + Random Forest + SVM
Running stack: BiLSTM + Extra Trees + SVM
Running stack: BiLSTM + Random Forest + XGBoost + SVM
Running stack: BiLSTM + Extra Trees + XGBoost + SVM
Running stack: BiLSTM + Random Forest + Extra Trees + XGBoost
Running stack: Random Forest + Extra Trees + XGBoost + SVM
Running stack: BiLSTM + Random Forest + Extra Trees + SVM


,Stack-base Model,Base_Models,Selected_Features,Train_ACC,Train_BACC,Train_PRE,Train_SPEC,Train_SENS,Train_F1,Train_MCC,Train_AUC,Test_ACC,Test_BACC,Test_PRE,Test_SPEC,Test_SENS,Test_F1,Test_MCC,Test_AUC
0,Random Forest + Extra Trees + XGBoost + SVM,Random Forest + Extra Trees + XGBoost + SVM,352,0.8807,0.8919,0.8370,0.9694,0.8919,0.8588,0.8104,0.9768,0.8826,0.8963,0.8421,0.9701,0.8963,0.8632,0.8137,0.9790
1,Random Forest + XGBoost + SVM,Random Forest + XGBoost + SVM,352,0.8798,0.8909,0.8360,0.9691,0.8909,0.8581,0.8088,0.9758,0.8831,0.8945,0.8400,0.9701,0.8945,0.8614,0.8139,0.9790
2,Random Forest + Extra Trees + SVM,Random Forest + Extra Trees + SVM,352,0.8771,0.8915,0.8343,0.9686,0.8915,0.8567,0.8055,0.9751,0.8790,0.8941,0.8409,0.9692,0.8941,0.8610,0.8086,0.9775
3,Random Forest + Extra Trees + XGBoost,Random Forest + Extra Trees + XGBoost,352,0.8806,0.8894,0.8357,0.9691,0.8894,0.8574,0.8095,0.9762,0.8849,0.8936,0.8443,0.9700,0.8936,0.8638,0.8155,0.9790
4,BiLSTM + Random Forest + XGBoost + SVM,BiLSTM + Random Forest + XGBoost + SVM,352,0.8797,0.8907,0.8356,0.9691,0.8907,0.8578,0.8086,0.9758,0.8824,0.8935,0.8389,0.9700,0.8935,0.8603,0.8129,0.9790
5,Extra Trees + XGBoost + SVM,Extra Trees + XGBoost + SVM,352,0.8806,0.8900,0.8363,0.9693,0.8900,0.8578,0.8099,0.9767,0.8835,0.8934,0.8417,0.9702,0.8934,0.8619,0.8145,0.9786
6,BiLSTM + Random Forest + Extra Trees + SVM,BiLSTM + Random Forest + Extra Trees + SVM,352,0.8770,0.8916,0.8336,0.9686,0.8916,0.8563,0.8054,0.9753,0.8784,0.8934,0.8380,0.9690,0.8934,0.8591,0.8075,0.9775
7,BiLSTM + Extra Trees + XGBoost + SVM,BiLSTM + Extra Trees + XGBoost + SVM,352,0.8799,0.8899,0.8360,0.9691,0.8899,0.8575,0.8090,0.9767,0.8842,0.8933,0.8424,0.9703,0.8933,0.8622,0.8155,0.9786
8,BiLSTM + Random Forest + XGBoost,BiLSTM + Random Forest + XGBoost,352,0.8808,0.8890,0.8345,0.9690,0.8890,0.8570,0.8094,0.9754,0.8842,0.8928,0.8432,0.9698,0.8928,0.8630,0.8142,0.9790
9,BiLSTM + Random Forest + Extra Trees + XGBoost,BiLSTM + Random Forest + Extra Trees + XGBoost,352,0.8810,0.8892,0.8353,0.9692,0.8892,0.8572,0.8100,0.9763,0.8840,0.8922,0.8432,0.9698,0.8922,0.8626,0.8141,0.9786


In [5]:
results_df.to_csv(RESULTS_PATH, index=False)
print(f'Saved results to {RESULTS_PATH}')

best_config = {
    'Stack-base Model': best_bundle['Stack-base Model'],
    'Base_Models': best_bundle['Base_Models'],
    'Selected_Features': best_bundle['Selected_Features'],
    'Test_BACC': float(best_bundle['metrics']['Test']['BACC']),
    'Test_ACC': float(best_bundle['metrics']['Test']['ACC']),
    'Test_F1': float(best_bundle['metrics']['Test']['F1']),
    'Test_AUC': float(best_bundle['metrics']['Test']['AUC']),
}
with open(BEST_CONFIG_PATH, 'w', encoding='utf-8') as handle:
    json.dump(best_config, handle, indent=2)
print(f'Saved best config to {BEST_CONFIG_PATH}')

best_predictions = pd.DataFrame({
    'y_true': y_test,
    'y_pred': best_bundle['test_pred'],
    'model_name': best_bundle['Stack-base Model'],
})
for class_index, class_name in enumerate(CLASS_NAMES):
    best_predictions[f'proba_{class_name}'] = best_bundle['test_proba'][:, class_index]
best_predictions.to_csv(BEST_PRED_PATH, index=False)
print(f'Saved best predictions to {BEST_PRED_PATH}')

best_bundle

Saved results to /Users/parvgoyal/IP/Multitaste-Model/fs_results/stacked_l1_results.csv
Saved best config to /Users/parvgoyal/IP/Multitaste-Model/fs_results/stacked_l1_best_config.json
Saved best predictions to /Users/parvgoyal/IP/Multitaste-Model/fs_results/stacked_l1_best_predictions.csv


{'Stack-base Model': 'Random Forest + Extra Trees + XGBoost + SVM',
 'Base_Models': 'Random Forest + Extra Trees + XGBoost + SVM',
 'Selected_Features': 352,
 'train_meta': array([[9.9666667e-01, 0.0000000e+00, 3.3333334e-03, ..., 6.4101943e-04,
         6.7493133e-06, 1.2372442e-03],
        [9.5333332e-01, 3.6666665e-02, 0.0000000e+00, ..., 9.3269417e-05,
         3.0527369e-04, 1.9564785e-03],
        [4.3689898e-01, 1.6780831e-01, 0.0000000e+00, ..., 1.2040834e-03,
         2.0223478e-02, 4.8749262e-01],
        ...,
        [9.8333335e-01, 9.9999998e-03, 0.0000000e+00, ..., 9.5556519e-05,
         8.1140235e-05, 6.9382100e-04],
        [9.7333336e-01, 2.6666667e-02, 0.0000000e+00, ..., 8.6594380e-05,
         8.0792335e-05, 1.0187016e-03],
        [9.8000002e-01, 9.9999998e-03, 0.0000000e+00, ..., 1.2782266e-04,
         2.0503454e-04, 6.7457091e-04]], shape=(13365, 20), dtype=float32),
 'test_meta': array([[9.86000000e-01, 1.40000000e-02, 0.00000000e+00, ...,
         1.00886979e